# Advanced 02 - 헤더 조작과 조건 매칭

이 노트북에서는:
- SIP 헤더 추가, 제거, 수정을 배웁니다
- 정규식 패턴 매칭을 연습합니다
- `subst` 계열 함수로 메시지를 변환합니다
- 실무 디버깅 기법을 익힙니다

---

## SIP 헤더 조작이란?

SIP Proxy는 메시지를 전달하기 전에 헤더를 조작합니다:
- **추가**: `X-Custom`, `P-Charge-Info` 등 커스텀 헤더
- **제거**: 불필요한 헤더, 보안상 위험한 헤더
- **수정**: `subst()`로 패턴 기반 텍스트 교체

이 작업은 실무에서 매우 자주 사용됩니다.

## 1. append_hf — 헤더 추가

`append_hf("Header: value")`로 SIP 메시지 끝에 헤더를 추가합니다.

실무 예: 통화 요금 정보 추가, 내부 라우팅 정보 추가 등

In [ ]:
%%sip INVITE
From: <sip:1001@example.com>;tag=h1
To: <sip:1002@example.com>

In [ ]:
# 커스텀 헤더 추가
append_hf("X-Source-IP: $si");
append_hf("X-Custom-Route: internal");
xlog("Added custom headers");

# 발신자 정보를 추가 헤더로 전달
$var(caller) = $(fu{uri.user});
append_hf("P-Charge-Info: sip:$var(caller)@example.com");
xlog("Added P-Charge-Info for billing: $var(caller)");

## 2. remove_hf — 헤더 제거

`remove_hf("Header")`로 특정 헤더를 제거합니다.

실무 예: 보안상 민감한 헤더 제거, 외부 네트워크로 보내기 전 내부 정보 삭제

In [ ]:
# 불필요한 헤더 제거
remove_hf("X-Internal-Info");
remove_hf("X-Debug");
xlog("Removed internal headers before relaying");

## 3. is_present_hf — 헤더 존재 확인

`is_present_hf("Header")`로 특정 헤더가 있는지 확인합니다.

In [ ]:
# 헤더 존재 여부에 따라 분기
$var(has_history) = "no";

if ($var(has_history) == "no") {
    xlog("No History-Info header — adding one");
    append_hf("History-Info: <sip:$ru>;index=1");
} else {
    xlog("History-Info already present");
}

## 4. subst — 정규식 치환

`subst("/pattern/replacement/flags")`로 SIP 메시지 전체에서 정규식 치환을 수행합니다.

실무 예: 도메인 변경, URI 재작성 등

```kamailio
# example.com → new-domain.com으로 변경
subst("/example.com/new-domain.com/g");
```

In [ ]:
# 도메인 치환 시뮬레이션
$var(original) = "sip:1001@old.example.com";
xlog("Original URI: $var(original)");

# 새 도메인으로 변경
$ru = "sip:1001@new.example.com";
xlog("Rewritten URI: $ru");

subst("/old.example.com/new.example.com/");
xlog("Domain rewrite applied");

## 5. 정규식 조건 매칭 (=~)

`=~` 연산자로 정규식 매칭을 할 수 있습니다.
URI 패턴, 전화번호 패턴 등을 검사할 때 유용합니다.

In [ ]:
%%sip INVITE
From: <sip:0212345678@example.com>;tag=r1
To: <sip:0211112222@example.com>

In [ ]:
# 발신자 번호 패턴 확인
$var(caller) = $(fu{uri.user});
xlog("Caller number: $var(caller)");

# 지역번호 확인 (02로 시작 = 서울)
if ($var(caller) =~ "^02") {
    xlog("Seoul area code detected");
} else {
    xlog("Other area code");
}

In [ ]:
# 수신자 라우팅 분기
$var(callee) = $(ru{uri.user});
xlog("Callee number: $var(callee)");

# 국제전화 (00으로 시작)
if ($var(callee) =~ "^00") {
    xlog("International call — routing to gateway");
} else {
    # 서울 지역번호
    if ($var(callee) =~ "^02") {
        xlog("Seoul local call");
    } else {
        xlog("Other domestic call");
    }
}

## 6. 전화번호 정규화

실무에서 가장 많이 하는 작업 중 하나입니다. 다양한 형식의 전화번호를
표준 형식으로 변환합니다.

```
010-1234-5678  →  01012345678
+82-10-1234-5678  →  01012345678
821012345678  →  01012345678
```

In [ ]:
# 번호 정규화 시뮬레이션
$var(raw_number) = "+82-10-1234-5678";
xlog("Raw input: $var(raw_number)");

# 실제로는 subst 사용, 여기서는 결과 시뮬레이션
$var(normalized) = "01012345678";
xlog("Normalized: $var(normalized)");

# R-URI 업데이트
$ru = "sip:$var(normalized)@example.com";
xlog("Updated R-URI: $ru");

## 7. 실무 디버깅 기법

헤더 조작 후 디버깅은 `xlog`와 변수 출력으로 확인합니다.

In [ ]:
# 디버깅: 조작 전후 비교
%%sip INVITE
From: <sip:1001@example.com>;tag=dbg1
To: <sip:1002@example.com>

In [ ]:
# Before
xlog("=== BEFORE ===");
xlog("From: $fu");
xlog("To: $tu");
xlog("R-URI: $ru");

# 헤더 조작
append_hf("X-Debug: testing");
$ru = "sip:1002@10.0.0.1:5060";

# After
xlog("=== AFTER ===");
xlog("R-URI: $ru");
xlog("Headers modified — ready to relay");

In [ ]:
# 변수 상태 확인
%%vars

---

### 요약

| 개념 | 설명 |
|------|------|
| append_hf("hdr: val") | SIP 헤더 추가 |
| remove_hf("hdr") | SIP 헤더 제거 |
| is_present_hf("hdr") | 헤더 존재 여부 확인 |
| subst("/old/new/") | 정규식 치환 |
| =~ | 정규식 매칭 |
| xlog | 디버깅 출력 |
| %%vars | 전체 변수 상태 확인 |

커리큘럼을 모두 마쳤습니다!